# Reproducing Tingley & Buzsáki (2018) from dandiset 000213

**Paper.** Tingley D, Buzsáki G (2018). Transformation of a spatial map across the
hippocampal-lateral septal circuit. *Neuron* 98, 1229-1242.
[doi:10.1016/j.neuron.2018.04.028](https://doi.org/10.1016/j.neuron.2018.04.028). This attempt used
the full text of the NIH author manuscript in PubMed Central (PMC6605060): Results, figure legends
and STAR Methods. The supplementary figures were not read.

**Dandiset.** [DANDI:000213](https://dandiarchive.org/dandiset/000213/0.250624.0438), version
0.250624.0438. The paper cites no version (its data statement says the data were being prepared for
CRCNS). There are two published versions, 0.220127.1738 and 0.250624.0438; they hold the same 67
files with the same asset identifiers and sizes (checked below) and differ only in dandiset-level
metadata, so the most recent is pinned.

**What the dandiset holds.** 67 sessions from the five rats (DT2, DT5, DT7, DT8, DT9): sorted units
labelled by region (`ls`, `hpc`, `ca3` and a few others), LFP at 1250 Hz, head position, and trials.
The paper's processed files (linearized behaviour, trial conditions, the CA1 channel of each session)
are not in it; they are rebuilt here as described in `lshpc.py`.

**Code read.** The paper's figure scripts, https://github.com/DavidTingley/papers (LS_phasecoding,
commit 9f4cd48a44f211956954558b66d776ccbb74d693); buzcode, https://github.com/buzsakilab/buzcode
(commit e615663b2b100782d1f378580f6d7aaa35c12937); and the per-cell decoder in
https://github.com/DavidTingley/scratch (commit 2f804532290774f52a442bf0f3ec23dd5398d58d). The
functions that bin phases (bz_phaseMap2Bins), compute information (olypherInfo) and detect
assemblies are not public.

**Claims.** The claims file is `claims/000213/10.1016-j.neuron.2018.04.028.yaml`.

| Claim | Tested here |
|---|---|
| ls-neuron-count: 1,647 LS neurons, 62 sessions | Tested |
| hpc-unit-count: 2,297 CA1 and 654 CA3 units | Tested |
| ls-active-neurons: 840 LS neurons fire at least 1.5 spikes per trial | Tested |
| fig2-ls-phase-vs-shuffle: 89% of LS neurons code position in phase | Tested |
| fig2-ls-phase-beats-rate: phase beats rate in 786/840 neurons, 3,046/4,393 conditions, 62/62 sessions | Tested |
| fig2e-ls-phase-lower-mse: phase MSE below rate MSE across LS conditions and sessions | Tested |
| fig2fg-mse-difference: mean phase minus rate MSE, LS and hippocampus | Tested |
| fig1i-hpc-rate-phase-equivalent: rate and phase decoding equivalent for place fields | Tested |
| ls-few-place-fields: 5.3% of LS and 31.7% of hippocampal rate maps have place fields | Tested |
| hpc-place-field-properties: 711 of 1,425 neurons with fields, 8.5 Hz, 38 bins | Tested |
| ls-place-fields-unstable: LS fields less stable (41.5 against 10) | Tested |
| fig1e-phase-info-smaller-scale: peak information at 15 (phase) and 29 (rate) bins | Tested |
| fig1g-phase-more-info-fine-scale: phase more informative below 6 cm, rate above 10 cm | Tested |
| fig4d-depth-gradient: phase coding grows with LS depth | Tested |
| fig4e-onset-depth: onset phase correlates with depth | Tested |
| fig6c-ca3-coherence-increases: LS-CA3 theta coherence rises along the run | Tested |
| fig6d-cfc-stronger-ca3: theta-gamma coupling rises, stronger for CA3 | Tested |
| figs7-ensemble-ls-phase-best: LS ensemble phases beat hippocampal ensembles | Tested |
| fig5-assemblies: 18.5% of pairs form gamma assemblies | Blocked: not runnable |
| fig5g-assembly-neurons-stronger-phase | Blocked: not runnable (depends on fig5-assemblies) |
| fig6e-assembly-position | Blocked: not runnable (depends on fig5-assemblies) |
| speed-independent: 2.8% of conditions speed-related | Blocked: methods underspecified |
| ls-information-lower: 0.4 against 1.7 bits/cm | Blocked: methods underspecified |
| figs6-reference-frame: maze rotation | Blocked: metadata insufficient |
| fig7-dynamic-weighting-model: spiking model | Blocked: other (a simulation) |

The whole notebook streams the data and takes about ten minutes on four processes, most of it the
per-cell decoding.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import lshpc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
results = lshpc.run_all(workers=4)
print(f"streamed and analysed {len(results)} files of dandiset {lshpc.DANDISET_ID} "
      f"version {lshpc.DANDISET_VERSION} in {time.time() - started:.0f} s")
errors = [(r["summary"]["path"], r["error"]) for r in results if "error" in r]
print("files that failed:", errors if errors else "none")

S = pd.DataFrame([r["summary"] for r in results])
S["session"] = S.index
S["name"] = S.session_id.str.slice(0, 32)
C = pd.DataFrame([dict(c, session=i, subject=results[i]["summary"]["subject"])
                  for i, r in enumerate(results) for c in r.get("cells", [])])
C["neuron"] = list(zip(C.session, C.unit))
A = C[C.active].copy()                 # included cell and condition pairs
LS = A[A.region == "ls"]
HP = A[A.region == "hpc"]
LSd = LS[LS.mse_phase.notna()]          # decoded (session has a theta channel)
HPd = HP[HP.mse_phase.notna()]

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
from dandi.dandiapi import DandiAPIClient

with DandiAPIClient() as client:
    ds = client.get_dandiset(lshpc.DANDISET_ID, lshpc.DANDISET_VERSION)
    meta = ds.get_raw_metadata()
    versions = [v.identifier for v in client.get_dandiset(lshpc.DANDISET_ID).get_versions()]
    listing = {}
    for v in versions:
        if v == "draft":
            continue
        listing[v] = {a.path: (a.identifier, a.size) for a in client.get_dandiset(lshpc.DANDISET_ID, v).get_assets()}
summary = meta["assetsSummary"]
print("Published versions:", [v for v in versions if v != "draft"])
pub = sorted(listing)
same = all(listing[pub[0]] == listing[v] for v in pub[1:])
print(f"Assets identical (path, asset id, size) across published versions: {same}; files: {len(listing[pub[0]])}")
print("Paper named by the dandiset:", [r.get("identifier") for r in meta.get("relatedResource", [])])
print("assetsSummary: subjects", summary.get("numberOfSubjects"), "| species", [s["name"] for s in summary.get("species", [])],
      "| variables", summary.get("variableMeasured"))
print("Subjects in the files:", sorted(S.subject.unique()), "| files per subject:", S.subject.value_counts().sort_index().to_dict())

The two published versions (2022 and 2025) hold the same 67 files with the same asset identifiers
and sizes; they differ only in dandiset-level metadata (contributor identifiers and affiliations,
the citation). The newest is pinned. The archive's summary of five rats of one species matches the
files. The paper's Methods state five rats, 3 male and 2 female; the files give no sex, so that
cannot be checked.

In [ ]:
units = (pd.DataFrame([dict(subject=s, location=l, n=n) for s, d in zip(S.subject, S.unit_locations) for l, n in d.items()])
         .pivot_table(index="subject", columns="location", values="n", aggfunc="sum", fill_value=0))
units.loc["all"] = units.sum()
print("Units by region label and subject:")
print(units.to_string())
sess = pd.DataFrame([dict(subject=s, location=l) for s, d in zip(S.subject, S.unit_locations) for l in d])
print("\nSessions holding units of each label:", sess.location.value_counts().to_dict())
print("Files without units:", S[S.n_units == 0].session_id.tolist())
print("Units with spike times not in increasing order:", int(S.units_unsorted.sum()),
      "in", S[S.units_unsorted > 0].session_id.tolist())
print("Units with spikes after the end of the LFP:", int(S.spikes_after_lfp_end.sum()))
print("\nElectrode labels per subject:")
for subj, g in S.groupby("subject"):
    labs = {}
    for d in g.electrode_locations:
        for k, v in d.items():
            labs[k] = labs.get(k, 0) + 1
    print(f"  {subj}: files with each label {labs}")
key = S[["n_spikes_total", "n_units"]].astype(str).agg("|".join, axis=1) + "|" + S.lfp_shape.astype(str)
print("\nFiles sharing total spikes, units and LFP shape:", S[key.duplicated(keep=False)].session_id.tolist())
print("Files sharing identical trial start times:", S[S.trial_starts_rounded.duplicated(keep=False)].session_id.tolist())
depth_rows = S[["subject", "session_id", "ls_depth", "hpc_depth", "has_hpc_units", "has_ca3_units"]]
print("\nDepths in the session names (um), first and second, with the hippocampal labels of the units:")
print(depth_rows.to_string(index=False))

Units carry a region label: `ls` (lateral septum), `hpc`, `ca3`, and a few cortical or other labels
(`acg`, `cc`, `ms`, `ppc`, `sFi`). One file (DT7, 20170317) has units without any label and is left
out of every count by region; one (DT7, 20170324) has no units. No two files share their spike
totals, unit counts and LFP length, or their trial times, so no recording was deposited twice.
Several sessions were recorded at the same depths on consecutive days (for example five DT8
sessions at 216 and 1944 um); each session's units are treated as separate units, as the paper's
counts do, and no claim assumes the same unit across sessions.

The electrode labels of sub-DT2 are all `unknown`; its hippocampal and LS channels are taken from
the electrode groups of its labelled units. The session names hold two depths that no metadata
describes. In DT9 the units are labelled `ca3` exactly in the sessions whose second depth is 828 um
or more, which fits the second depth being the hippocampal probe's; the first depth is taken as the
LS probe's. This is an assumption, used only by the depth claims.

In [ ]:
pos = S[["subject", "name", "task", "position_unit_stated", "position_extent_stated_units", "position_unit_used",
         "position_dt_median", "position_dt", "position_nonincreasing_steps", "position_z_equals_y",
         "position_nan_fraction", "path_length_median"]].copy()
pos["extent_1_99"] = pos.position_extent_stated_units.round(2)
print(pos.drop(columns="position_extent_stated_units").round(4).to_string(index=False))
odd = S[(S.position_unit_stated != "pixels") & (((S.position_unit_stated == "mm") & (S.position_extent_stated_units < 10)) |
                                               ((S.position_unit_stated == "m") & (S.position_extent_stated_units > 10)))]
print(f"\nFiles whose stated position unit disagrees with the values: {len(odd)}:", odd.name.tolist())
print("Files whose third position column equals the second:", int(S.position_z_equals_y.sum()), "of", len(S))
print("Files with repeated or decreasing position timestamps:", S[S.position_nonincreasing_steps > 0].name.tolist())

Position is stated in `m`, `mm` or `pixels`. Eight DT8 files state `mm` but hold values spanning about
1.1 to 1.3, which are metres (a maze of about a metre across; the same rat's other files span 1000
to 1200 mm). The unit used here is inferred from the extent: below 10 the values are taken as
metres, otherwise as millimetres; sub-DT2 positions stay in pixels, with no scale given. In every
file the third position column (z) is a copy of the second (y). In eight of the eleven sub-DT2 files
position timestamps repeat or go back (the count of such steps is printed), so the time per sample
is taken as the session's span over its sample count, which gives 33.3 ms (30 Hz) in every sub-DT2
file. The median path of a trial is 2.2 to 2.9 m on
the alternation mazes and 1.3 to 1.7 m on the linear track (outside sub-DT2), against the paper's
maze of about 3.2 m; the paper gives no other maze dimensions in the main text.

In [ ]:
tr = S[["subject", "name", "task", "n_trials", "trial_s_median", "trial_types", "directions", "conditions_used",
        "trials_without_positions", "trials_after_position_end"]]
print(tr.to_string(index=False))
print("\nTrial types:", sorted({str(t) for x in S.trial_types for t in x}), "| directions:", sorted({str(t) for x in S.directions for t in x}))
print("Files whose trials carry no type or direction:", S[S.trial_types.apply(lambda x: x == [""])].name.tolist())
print("Conditions with at least 10 trials, all files:", int(S.conditions_used.sum()))

The trials table has `trial_type` (central or wheel alternation) and `direction` (clockwise or
counter-clockwise) for the alternation tasks, and only start and stop times on the linear track and
in two DT8 alternation files (20170207 and 20170224).
Trials last a median of 1.6 to 5.2 s. The paper's trial conditions are not in the dandiset; the
conditions formed here (type, direction, start and end site) give 0 to 13 conditions of at least
10 trials per file. DT9 20170501 has none: its trials do not start or end at common sites (see the
clock check below).

In [ ]:
lf = S[["subject", "name", "lfp_shape", "lfp_chunks", "lfp_rate", "lfp_start", "lfp_conversion", "hpc_channels_from",
        "n_hpc_channels", "n_ls_channels", "theta_channel", "ls_channel"]].copy()
lf["theta_ratio"] = [d["theta_ratio"] if isinstance(d, dict) else np.nan for d in S.theta_channel_selection]
lf["median_ratio"] = [d["median_theta_ratio"] if isinstance(d, dict) else np.nan for d in S.theta_channel_selection]
lf["lfp_conversion"] = lf.lfp_conversion.map("{:.3g}".format)
print(lf.round(2).to_string(index=False))
print("\nLFP sampling rates:", S.lfp_rate.unique(), "| starting times:", S.lfp_start.unique())
print("Files without a hippocampal channel (no theta reference):", S[~S.has_theta].name.tolist())

The LFP is stored at 1250 Hz from time 0 in every file, as the paper's code assumes
(`phases(round(t*1250))`). 42 files chunk it as 500 samples by all channels, so the analysis reads
only the trials (with 1 s on each side), two channels at a time. One file (DT2, 20160217) has no
hippocampal units or labelled hippocampal channels, so it has no theta reference: its LS units are
counted, but not decoded.

In [ ]:
lags = list(lshpc.TRIAL_SHIFTS_S)
sp = pd.DataFrame([r["summary"]["clock_trial_speed"] for r in results], index=S.name)
sp_peak = sp.idxmax(axis=1)
print("Mean running speed (position units per s) inside the trial windows shifted by each lag (s); peak lag per file:")
print(pd.concat([sp.round(1), sp_peak.rename("peak")], axis=1).to_string())
print("\nPeak lag of speed inside trials, files per lag:", sp_peak.value_counts().sort_index().to_dict())
ratio = sp[0.0] / sp[[-3.0, 3.0]].mean(axis=1)
print("Speed at lag 0 over speed at +-3 s, median", round(float(ratio.median()), 2), "| range", round(float(ratio.min()), 2), "to", round(float(ratio.max()), 2))

lfp_lags = list(lshpc.LFP_LAGS_MS)
lk = pd.DataFrame([r.get("clock_lfp", {}) for r in results], index=S.name)
ok = lk.notna().all(axis=1)
lk_peak = lk[ok].idxmax(axis=1)
print("\nTheta phase locking (mean resultant length) of hippocampal spikes, LFP shifted by each lag (ms):")
print(pd.concat([lk[ok].round(3), lk_peak.rename("peak")], axis=1).to_string())
print("Files per peak lag (ms):", lk_peak.value_counts().sort_index().to_dict())
print("Locking at 0 ms over locking at +-200 ms, median",
      round(float((lk[ok][0] / lk[ok][[-200, 200]].mean(axis=1)).median()), 2))

rt = pd.DataFrame([r["summary"]["clock_trial_rate"] or {} for r in results], index=S.name).dropna()
rt_peak = rt.idxmax(axis=1)
print("\nMean firing rate of all units in the trial windows shifted by each lag, files per peak lag:",
      rt_peak.value_counts().sort_index().to_dict())

Three clock checks, each a contrast between the true and shifted alignments.

- Positions against the trials table: running speed inside the trial windows peaks at a lag of 0
  or 0.5 s in 65 of 67 files and is a median 2.2 times its value at 3 s, so positions and trials share a
  clock to within about half a second. In DT9 20170501 speed does not change with the lag (31.5
  cm/s at 0 against 29.1 to 32.8 at other lags): its trials do not match its positions, and it is
  left out by the condition rule (no condition reaches 10 trials). In one DT2 file (20160221) speed
  peaks at a lag of 1 s; its trials may be offset from its positions by up to a second. This is
  consistent with the data, not proven for every trial.
- Spikes against the LFP: theta phase locking of hippocampal spikes peaks within 75 ms of zero lag
  in 44 of the 47 files with hippocampal units and a theta channel, and at 200 ms it is lower by a
  median of about a sixth (ratio 1.2). This is consistent with a shared clock to
  within a fraction of a theta cycle. It could not be checked in the 17 DT8 files and the three DT2 and
  DT7 files without labelled hippocampal units.
- Spikes against the trials: the mean rate of all units in the trial windows changes little with
  the lag (running modulates many LS units weakly or negatively), so this contrast does not settle
  the alignment of spikes and trials. It is not established for the LS-only sessions beyond the
  spike-LFP check of the other files.

## LS neurons recorded (ls-neuron-count)

> A total of 1,647 LS neurons were recorded from five animals, in 62 sessions.

In [ ]:
ls_units = units["ls"]
print("LS units per subject:", ls_units.drop("all").to_dict(), "| total", int(ls_units["all"]))
n_ls_sessions = sum("ls" in d for d in S.unit_locations)
print("Sessions with LS units:", n_ls_sessions, "| subjects with LS units:", int((ls_units.drop("all") > 0).sum()))
print("Paper: 1,647 LS neurons, 62 sessions, five animals. Difference in units:",
      int(ls_units["all"]) - 1647, f"({(int(ls_units['all']) - 1647) / 1647:+.1%})")
print("Unlabelled units (DT7 20170317), not counted:", int(units.loc["all"].get("unlabelled", 0)))

**Result: reproduced.** 1,696 units are labelled LS, 3.0% more than the paper's 1,647 and within tolerance, in 62 sessions as in the paper, and all five rats hold LS units.

## Hippocampal units (hpc-unit-count)

> The hippocampal recordings hold 2,297 isolated CA1 units and 654 CA3 units (2,951 in all).

In [ ]:
ca1, ca3 = int(units.loc["all", "hpc"]), int(units.loc["all", "ca3"])
print("Units labelled hpc (taken as CA1) per subject:", units["hpc"].drop("all").to_dict(), "| total", ca1)
print("Units labelled ca3 per subject:", units["ca3"].drop("all").to_dict(), "| total", ca3)
print(f"CA1: {ca1} against 2,297 ({(ca1 - 2297) / 2297:+.1%}); CA3: {ca3} against 654 ({(ca3 - 654) / 654:+.1%}); "
      f"sum {ca1 + ca3} against 2,951 ({(ca1 + ca3 - 2951) / 2951:+.1%})")
print("Sessions with hpc units:", int(S.has_hpc_units.sum()) - int(S.has_ca3_units.sum()), "| with ca3 units:", int(S.has_ca3_units.sum()),
      "| paper, Figure 6A: 63 CA1 and 19 CA3 sessions; 45 hippocampal sessions in Figure 1I")

**Result: partially reproduced.** 2,593 units are labelled hpc against 2,297 CA1 units (+12.9%, outside tolerance) and 624 are labelled ca3 against 654 (-4.6%, within). Their sum, 3,217, is 9.0% above 2,951. The labels hpc and ca3 are read as CA1 and CA3; the dandiset does not say that hpc means CA1.

## LS neurons passing the activity criterion (ls-active-neurons)

> Of the LS neurons, 840 fired at least 1.5 action potentials per trial on average.

In [ ]:
ls_all = C[C.region == "ls"]
def active_neurons(df, k):
    return df[df.n_spikes >= k * df.n_trials].groupby("neuron").ngroups
n_active = LS.neuron.nunique()
print("LS units in a condition with at least 10 trials:", ls_all.neuron.nunique(), "of", int(units.loc["all", "ls"]))
print("Active LS neurons (1.5 spikes per trial in at least one condition):", n_active, "| paper 840")
print("Per subject:", LS.groupby("subject").neuron.nunique().to_dict())
print("Included LS conditions (neuron and condition pairs):", len(LS), "| paper 4,393 conditions")
for k in (1.0, 1.5, 2.0, 3.0):
    print(f"  threshold {k} spikes per trial: {active_neurons(ls_all, k)} neurons")

**Result: reproduced.** 793 LS neurons fire at least 1.5 spikes per trial in a condition with at least 10 trials, against 840 (-5.6%, within tolerance), over 3,694 neuron-condition pairs (the paper has 4,393). The count depends on how conditions are formed, which the dandiset does not record.

## Phase coding against shuffled data (fig2-ls-phase-vs-shuffle)

> The majority of LS neurons (751 of 840, 89%) showed position encoding in their spike theta phase,
> compared with shuffled data, in at least one condition.

In [ ]:
def frac_neurons(df, mask):
    return mask.groupby(df.neuron).any()

sig_chance = (LSd.p_phase_vs_chance < 0.01) & (LSd.mse_phase < LSd.mse_chance_phase)
per_neuron = frac_neurons(LSd, sig_chance)
print(f"Active LS neurons decoded: {LSd.neuron.nunique()} (dropped {n_active - LSd.neuron.nunique()} in the file without theta reference)")
print(f"Neurons with phase MSE below chance at p < 0.01 in at least one condition: {int(per_neuron.sum())} "
      f"of {len(per_neuron)} = {per_neuron.mean():.3f} | paper 751 of 840 = 0.894")
print("Conditions significant:", f"{sig_chance.mean():.3f}", "of", len(LSd))
print("Per subject:", per_neuron.groupby([S.subject[n[0]] for n in per_neuron.index]).mean().round(3).to_dict())
for p in (0.05, 0.01, 0.001):
    m = (LSd.p_phase_vs_chance < p) & (LSd.mse_phase < LSd.mse_chance_phase)
    print(f"  p < {p}: {frac_neurons(LSd, m).mean():.3f} of neurons")
print("Mean MSE (bins^2) over LS conditions: phase", round(LSd.mse_phase.mean()), "| chance phase", round(LSd.mse_chance_phase.mean()),
      "| rate", round(LSd.mse_rate.mean()), "| chance rate", round(LSd.mse_chance_rate.mean()),
      "| uniform guessing over 200 bins", round(2 * (200**2 - 1) / 12))
rate_sig = (LSd.p_rate_vs_chance < 0.01) & (LSd.mse_rate < LSd.mse_chance_rate)
print(f"For comparison, rate MSE below chance at p < 0.01: {frac_neurons(LSd, rate_sig).mean():.3f} of neurons")

**Result: not reproduced.** 52.1% of the decoded active LS neurons (408 of 783) decode position from phase better than chance at p < 0.01 in at least one condition, against 89% in the paper. With p < 0.05 the fraction is 66.0%, still outside tolerance; at p < 0.01 the rate code passes the same test against its own chance in 54.4% of neurons. The decoder and the phase maps are rebuilt from the Methods and code, without the paper's linearization, conditions or theta channel.

## Phase decoding better than rate decoding (fig2-ls-phase-beats-rate)

> Spike-phase decoding outperformed rate decoding of the rat's position for 786 of 840 LS neurons,
> 3,046 of 4,393 conditions, and 62 of 62 sessions.

In [ ]:
better = (LSd.p_phase_vs_rate < 0.01) & (LSd.mse_phase < LSd.mse_rate)
nb = frac_neurons(LSd, better)
ses = LSd.groupby("session")[["mse_phase", "mse_rate"]].mean()
print(f"Neurons: {int(nb.sum())} of {len(nb)} = {nb.mean():.3f} | paper 786 of 840 = 0.936")
print(f"Conditions: {int(better.sum())} of {len(better)} = {better.mean():.3f} | paper 3,046 of 4,393 = 0.693")
print(f"Sessions with lower mean phase than rate MSE: {int((ses.mse_phase < ses.mse_rate).sum())} of {len(ses)} = "
      f"{(ses.mse_phase < ses.mse_rate).mean():.3f} | paper 62 of 62")
worse = (LSd.p_phase_vs_rate < 0.01) & (LSd.mse_phase > LSd.mse_rate)
print(f"Conditions where rate is better at p < 0.01: {worse.mean():.3f}")
by = pd.DataFrame({"neurons": nb.groupby([S.subject[n[0]] for n in nb.index]).mean(),
                   "conditions": better.groupby(LSd.subject).mean(),
                   "sessions": (ses.mse_phase < ses.mse_rate).groupby(S.subject[ses.index].values).mean()})
print("\nPer subject (fractions):")
print(by.round(3).to_string())
for p in (0.05, 0.001):
    m = (LSd.p_phase_vs_rate < p) & (LSd.mse_phase < LSd.mse_rate)
    print(f"  p < {p}: neurons {frac_neurons(LSd, m).mean():.3f}, conditions {m.mean():.3f}")
bt = (LSd.p_phase_vs_rate_trialsplit < 0.01) & (LSd.mse_phase_trialsplit < LSd.mse_rate_trialsplit)
print(f"Variant, whole trials held out: neurons {frac_neurons(LSd, bt).mean():.3f}, conditions {bt.mean():.3f}")

**Result: not reproduced.** Phase beats rate at p < 0.01 in 84.5% of neurons (paper 93.6%) and 48.1% of conditions (paper 69.3%), both outside tolerance, and in 58 of 58 sessions (paper 62 of 62). As fewer than half the conditions pass, the criterion's partial outcome does not apply. The direction holds in every rat and session.

## Phase MSE lower than rate MSE (fig2e-ls-phase-lower-mse)

> Across LS neurons the spike-phase code is stronger than the rate code (two-sample t tests;
> N = 4,393 conditions, p < 1e-100; N = 62 sessions, p < 1e-10).

In [ ]:
t_c = stats.ttest_ind(LSd.mse_phase, LSd.mse_rate)
t_s = stats.ttest_ind(ses.mse_phase, ses.mse_rate)
print(f"Conditions (N = {len(LSd)}): phase {LSd.mse_phase.mean():.0f}, rate {LSd.mse_rate.mean():.0f}; t = {t_c.statistic:.1f}, p = {t_c.pvalue:.2e}")
print(f"Sessions (N = {len(ses)}): phase {ses.mse_phase.mean():.0f}, rate {ses.mse_rate.mean():.0f}; t = {t_s.statistic:.1f}, p = {t_s.pvalue:.2e}")
print("Per subject, conditions:")
for subj, g in LSd.groupby("subject"):
    t = stats.ttest_ind(g.mse_phase, g.mse_rate)
    print(f"  {subj}: N = {len(g)}, phase {g.mse_phase.mean():.0f}, rate {g.mse_rate.mean():.0f}, p = {t.pvalue:.1e}")
tt = stats.ttest_ind(LSd.mse_phase_trialsplit, LSd.mse_rate_trialsplit)
print(f"Variant, whole trials held out: phase {LSd.mse_phase_trialsplit.mean():.0f}, rate {LSd.mse_rate_trialsplit.mean():.0f}, p = {tt.pvalue:.1e}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
bins = np.arange(2000, 14001, 250)
ax[0].hist(LSd.mse_phase, bins, alpha=.6, label="phase"); ax[0].hist(LSd.mse_rate, bins, alpha=.6, label="rate")
ax[0].set_xlabel("MSE (bins$^2$)"); ax[0].set_ylabel("LS conditions"); ax[0].legend()
d = LSd.mse_phase - LSd.mse_rate
ax[1].hist(d, np.arange(-6000, 6001, 200), color="m"); ax[1].set_xlabel("MSE phase - rate (bins$^2$)")
plt.tight_layout(); plt.show()
print("Outside the axes: MSE", int(((LSd.mse_phase < 2000) | (LSd.mse_phase > 14000)).sum()), "phase and",
      int(((LSd.mse_rate < 2000) | (LSd.mse_rate > 14000)).sum()), "rate values; difference", int((d.abs() > 6000).sum()),
      "| ranges: phase", LSd.mse_phase.min().round(), "to", LSd.mse_phase.max().round(), ", rate", LSd.mse_rate.min().round(), "to", LSd.mse_rate.max().round())

**Result: reproduced.** Over 3,660 LS conditions the phase MSE (6,359) is below the rate MSE (7,404), t = -33.8, p = 2.5e-232, below the paper's 1e-100; over 58 sessions p = 2.9e-23, below 1e-10. It holds in each rat (DT7 only at p = 0.018) and with whole trials held out.

## Hippocampal place fields, rate and phase decoding (fig1i-hpc-rate-phase-equivalent)

> For hippocampal place fields the distributions of rate and phase decoding errors are
> statistically equivalent (two-sample KS test; 1,820 place fields, p = 0.19; 45 sessions, p = 0.61).

In [ ]:
PF = HPd[HPd.n_fields > 0]
hs = PF.groupby("session")[["mse_phase", "mse_rate"]].mean()
k1 = stats.ks_2samp(PF.mse_phase, PF.mse_rate)
k2 = stats.ks_2samp(hs.mse_phase, hs.mse_rate)
print(f"Place fields: N = {len(PF)} (paper 1,820); phase {PF.mse_phase.mean():.0f}, rate {PF.mse_rate.mean():.0f}; "
      f"KS D = {k1.statistic:.3f}, p = {k1.pvalue:.3f}; mean difference {(PF.mse_phase - PF.mse_rate).mean():.0f}")
print(f"Sessions: N = {len(hs)} (paper 45); KS D = {k2.statistic:.3f}, p = {k2.pvalue:.3f}; mean difference {(hs.mse_phase - hs.mse_rate).mean():.0f}")
print("Per subject (fields):")
for subj, g in PF.groupby("subject"):
    k = stats.ks_2samp(g.mse_phase, g.mse_rate)
    print(f"  {subj}: N = {len(g)}, phase - rate {(g.mse_phase - g.mse_rate).mean():.0f}, KS p = {k.pvalue:.3f}")
k3 = stats.ks_2samp(HPd.mse_phase, HPd.mse_rate)
print(f"Variant, all included hippocampal conditions (N = {len(HPd)}): phase - rate {(HPd.mse_phase - HPd.mse_rate).mean():.0f}, KS p = {k3.pvalue:.2e}")
k4 = stats.ks_2samp(PF.mse_phase_trialsplit, PF.mse_rate_trialsplit)
print(f"Variant, whole trials held out: phase - rate {(PF.mse_phase_trialsplit - PF.mse_rate_trialsplit).mean():.0f}, KS p = {k4.pvalue:.3f}")

**Result: reproduced.** The KS test finds no difference between phase and rate MSE for 315 hippocampal place fields (p = 0.115) or over 35 sessions (p = 0.874), as the paper reports. The phase MSE is on average 167 bins squared lower. Only 315 fields are found here, against 1,820 (see ls-few-place-fields); over all hippocampal conditions the distributions differ (p = 7.4e-48).

## Mean phase minus rate MSE (fig2fg-mse-difference)

> The mean phase minus rate MSE is about -1,000 for LS conditions and -33 for hippocampal place
> fields, and -1,024 and -25 for LS and hippocampal sessions.

In [ ]:
vals = {"LS conditions": (LSd.mse_phase - LSd.mse_rate).mean(), "hippocampal place fields": (PF.mse_phase - PF.mse_rate).mean(),
        "LS sessions": (ses.mse_phase - ses.mse_rate).mean(), "hippocampal sessions": (hs.mse_phase - hs.mse_rate).mean()}
paper = {"LS conditions": -1000, "hippocampal place fields": -33, "LS sessions": -1024, "hippocampal sessions": -25}
for k, v in vals.items():
    print(f"{k}: {v:.0f} (paper {paper[k]}, tolerance +-{abs(paper[k]) * 0.1:.1f})")
print("\nPer subject, LS conditions and hippocampal place fields:")
print(pd.DataFrame({"LS": (LSd.mse_phase - LSd.mse_rate).groupby(LSd.subject).mean(),
                    "HPC fields": (PF.mse_phase - PF.mse_rate).groupby(PF.subject).mean()}).round(0).to_string())
print("\nVariant, whole trials held out: LS", round((LSd.mse_phase_trialsplit - LSd.mse_rate_trialsplit).mean()),
      "| hippocampal fields", round((PF.mse_phase_trialsplit - PF.mse_rate_trialsplit).mean()))

**Result: partially reproduced.** The LS condition mean, -1,045, is within tolerance of -1,000; the LS session mean, -1,132, is just outside -1,024 +- 102; the hippocampal values, -167 for place fields and -221 for sessions, are far from -33 and -25. LS values are negative and larger in magnitude than the hippocampal ones at both levels.

## Place fields in LS and hippocampus (ls-few-place-fields)

> Only 5.3% (232 of 4,393) of LS rate maps showed place fields by the criteria applied to
> hippocampal units, where 31.7% (1,820 of 5,735) did.

In [ ]:
f_ls, f_hp = (LS.n_fields > 0), (HP.n_fields > 0)
print(f"LS rate maps with a field: {int(f_ls.sum())} of {len(LS)} = {f_ls.mean():.3f} (paper 0.053)")
print(f"Hippocampal rate maps with a field: {int(f_hp.sum())} of {len(HP)} = {f_hp.mean():.3f} (paper 0.317)")
tab = [[f_ls.sum(), (~f_ls).sum()], [f_hp.sum(), (~f_hp).sum()]]
chi = stats.chi2_contingency(tab)
print(f"Chi-squared {chi.statistic:.1f}, p = {chi.pvalue:.2e}")
print(pd.DataFrame({"LS": f_ls.groupby(LS.subject).mean(), "HPC": f_hp.groupby(HP.subject).mean(),
                    "n LS": f_ls.groupby(LS.subject).size(), "n HPC": f_hp.groupby(HP.subject).size()}).round(3).to_string())

**Result: not reproduced.** 5.8% of LS rate maps have a place field, within tolerance of 5.3%, but so do only 6.2% of hippocampal rate maps against 31.7%, and the two fractions do not differ (chi-squared p = 0.41). The hippocampal fraction is low in every rat (6.0 to 6.9%).

## Properties of hippocampal place fields (hpc-place-field-properties)

> 711 of 1,425 hippocampal neurons had at least one place field (1.27 fields per neuron on
> average); the median peak in-field rate was 8.5 Hz and the median field width 38 bins.

In [ ]:
has = (HP.n_fields > 0).groupby(HP.neuron).any()
peaks = np.concatenate(HP.field_peaks_hz.dropna().values)
widths = np.concatenate(HP.field_widths.dropna().values)
nf = HP.groupby("neuron").n_fields.sum()
print(f"Hippocampal neurons with a field: {int(has.sum())} of {len(has)} = {has.mean():.3f} (paper 711 of 1,425 = 0.499)")
print(f"Fields per active neuron: mean {nf.mean():.2f} (paper 1.27)")
print(f"Median peak rate {np.median(peaks):.1f} Hz (paper 8.5); median width {np.median(widths):.0f} bins (paper 38); N = {len(peaks)} fields")
print("Per subject: fraction with a field", has.groupby([S.subject[n[0]] for n in has.index]).mean().round(3).to_dict())

**Result: partially reproduced.** The median field width, 40 bins, is within tolerance of 38; but 14.6% of active hippocampal neurons have a field against 49.9%, with 0.29 fields per neuron against 1.27, and the median peak rate is 12.5 Hz against 8.5.

## Stability of LS and hippocampal place fields (ls-place-fields-unstable)

> The few LS rate maps with a place field were less stable across trials than CA1/CA3 place fields
> (median coefficient of variance 41.5 against 10; two-sample t test, p < 1e-20).

In [ ]:
lf_, hf_ = LS[LS.n_fields > 0], HP[HP.n_fields > 0]
for name, col, scale in (("(a) SD over trials at the field peak, Hz", "field_peak_sd", 1), ("(b) SD over mean at the peak, %", "field_peak_cv", 100)):
    a, b = lf_[col] * scale, hf_[col] * scale
    t = stats.ttest_ind(a, b)
    print(f"{name}: LS median {a.median():.1f} (N = {len(a)}), hippocampus median {b.median():.1f} (N = {len(b)}); t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
print("Paper: 41.5 against 10.")

**Result: partially reproduced.** Under reading (b), SD over mean at the field peak, LS fields are more variable than hippocampal ones (medians 88.7% and 84.6%, p = 3.4e-04), but the values are far from 41.5 and 10. Under reading (a), the code's SD at the peak, the hippocampal median of 10.5 Hz matches the paper's 10 but the LS median is lower, 7.0 Hz, the reverse of the claim.

## Smoothing scale of peak information (fig1e-phase-info-smaller-scale)

> Hippocampal place fields reach their peak information at smaller smoothing scales for phase than
> for rate (medians 15 and 29 bins; two-sample t test, p < 1e-40).

In [ ]:
info = [dict(x, session=i, subject=results[i]["summary"]["subject"]) for i, r in enumerate(results) for x in r.get("info", [])]
def smooth5(v):
    return lshpc.matlab_smooth(v, 5)
best_p = np.array([np.argmax(smooth5(x["phase"])) + 1 for x in info])
best_r = np.array([np.argmax(smooth5(x["rate"])) + 1 for x in info])
keep_p, keep_r = best_p < 49, best_r < 49
t = stats.ttest_ind(best_p[keep_p], best_r[keep_r])
print(f"Place fields with information curves: {len(info)}; kept (best smoothing below 49): phase {keep_p.sum()}, rate {keep_r.sum()}")
print(f"Median best smoothing: phase {np.median(best_p[keep_p]):.0f} bins, rate {np.median(best_r[keep_r]):.0f} bins (paper 15 and 29); "
      f"means {best_p[keep_p].mean():.1f} and {best_r[keep_r].mean():.1f}; t = {t.statistic:.2f}, p = {t.pvalue:.2e}")
print(f"Fields left out at 49 or more: phase {(~keep_p).sum()}, rate {(~keep_r).sum()}")
subj = np.array([x["subject"] for x in info])
for s_ in sorted(set(subj)):
    m = subj == s_
    print(f"  {s_}: N = {m.sum()}, median phase {np.median(best_p[m & keep_p]) if (m & keep_p).any() else np.nan}, "
          f"rate {np.median(best_r[m & keep_r]) if (m & keep_r).any() else np.nan}")
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(best_p[keep_p], np.arange(0, 51, 2), alpha=.6, label="phase"); ax.hist(best_r[keep_r], np.arange(0, 51, 2), alpha=.6, label="rate")
ax.set_xlabel("smoothing with peak information (bins)"); ax.set_ylabel("place fields"); ax.legend(); plt.show()
print("Plotted values: phase", best_p[keep_p].min(), "to", best_p[keep_p].max(), "| rate", best_r[keep_r].min(), "to", best_r[keep_r].max(), "bins; none outside the axes")

**Result: partially reproduced.** Phase information peaks at smaller smoothing than rate information (medians 10 and 19 bins, p = 4.9e-18), but both medians are below the paper's 15 and 29, and p does not reach 1e-40 with 315 fields.

## Information of phase and rate at fine and coarse scales (fig1g-phase-more-info-fine-scale)

> At smoothing below 6 cm (about one theta cycle) phase carries more spatial information than rate
> in hippocampal place fields, and above 10 cm rate carries more.

In [ ]:
P = np.array([x["phase"] / x["n_trials"] for x in info])
R = np.array([x["rate"] / x["n_trials"] for x in info])
rows = []
for w in lshpc.INFO_SMOOTHINGS:
    t = stats.ttest_ind(P[:, w - 1], R[:, w - 1])
    rows.append({"smoothing": w, "phase_minus_rate": (P[:, w - 1] - R[:, w - 1]).mean(), "p": t.pvalue})
D = pd.DataFrame(rows)
print(D[D.smoothing.isin([1, 2, 3, 4, 5, 6, 7, 10, 15, 20, 30, 40, 50])].to_string(index=False))
fine = D[D.smoothing.isin([2, 3])]
coarse = D[D.smoothing >= 7]
print(f"\nFine (2 and 3 bins): phase above rate at p < 1e-5 in {int(((fine.phase_minus_rate > 0) & (fine.p < 1e-5)).sum())} of 2")
print(f"Coarse (7 to 50 bins): rate above phase at p < 0.05 in {int(((coarse.phase_minus_rate < 0) & (coarse.p < 0.05)).sum())} of {len(coarse)} "
      f"({((coarse.phase_minus_rate < 0) & (coarse.p < 0.05)).mean():.2f})")
print("Bin size: the median trial path over 200 bins is", round(float(np.nanmedian(S[S.position_unit_used == "cm"].path_length_median)) / 200, 2),
      "cm in the files with a scale (paper about 1.5 cm)")

**Result: partially reproduced.** At coarse smoothing (7 to 50 bins) rate carries more information than phase at p < 0.05 in 44 of 44 smoothings, as the paper reports above 10 cm. At fine smoothing phase is not more informative at p < 1e-5 (2 bins, difference +0.015, p = 0.005; 3 bins, -0.001, p = 0.80). Bins here are about 1.1 cm, so 2 and 3 bins are below 6 cm.

## Phase coding and LS depth (fig4d-depth-gradient)

> The strength of phase coding relative to rate coding increases with recording depth within the LS,
> within sessions and across animals.

In [ ]:
sd = LSd.groupby("session").apply(lambda g: (g.mse_phase - g.mse_rate).mean()).rename("diff").to_frame()
sd["subject"] = S.subject[sd.index].values
sd["depth"] = S.ls_depth[sd.index].values
rows = []
for subj, g in sd.groupby("subject"):
    if len(g) >= 4 and g.depth.nunique() > 1:
        r, p = stats.pearsonr(g.depth, g["diff"])
    else:
        r, p = np.nan, np.nan
    rows.append({"subject": subj, "sessions": len(g), "depth_range": f"{g.depth.min()}-{g.depth.max()}", "r": r, "p": p})
print(pd.DataFrame(rows).round(3).to_string(index=False))
c = sd.copy()
c["depth"] -= c.groupby("subject").depth.transform("mean")
c["diff"] -= c.groupby("subject")["diff"].transform("mean")
r, p = stats.pearsonr(c.depth, c["diff"])
print(f"Pooled, centred within animal: r = {r:.3f}, p = {p:.3f}, N = {len(c)} sessions")
r2, p2 = stats.pearsonr(sd.depth, sd["diff"])
print(f"Variant, pooled without centring: r = {r2:.3f}, p = {p2:.3f}")
sd2 = sd.copy(); sd2["depth"] = S.hpc_depth[sd.index].values
for subj, g in sd2.groupby("subject"):
    if len(g) >= 4 and g.depth.nunique() > 1:
        print(f"  variant with the second depth, {subj}: r = {stats.pearsonr(g.depth, g['diff'])[0]:.3f}")

**Result: partially reproduced.** With depth and value centred within each rat, deeper LS sessions have more negative phase minus rate MSE (r = -0.365, p = 0.005, 58 sessions). Per rat r is negative in DT2 (-0.70), DT5 (-0.43) and DT9 (-0.62) but positive in DT8 (+0.24), so it does not hold in every rat. The within-session part cannot be tested, as the dandiset gives no recording site per unit.

## Onset phase of LS phase fields and depth (fig4e-onset-depth)

> The theta phase at the onset of LS phase fields varies with recording depth (DT5 r = 0.38, DT9 r = 0.25,
> DT2 r = 0.27; circular-linear correlation, p < 0.001).

In [ ]:
sig = LSd[sig_chance & LSd.onset_phase.notna()]
best = sig.loc[sig.groupby("neuron").mse_phase.idxmin()]
best = best.assign(depth=S.ls_depth[best.session].values)
print(f"Neurons with significant phase coding and an onset phase: {len(best)}")
for subj in ["DT5", "DT9", "DT2", "DT7", "DT8"]:
    g = best[best.subject == subj]
    if len(g) > 5 and g.depth.nunique() > 1:
        r, p = lshpc.circ_corrcl(g.onset_phase.values, g.depth.values)
        print(f"  {subj}: N = {len(g)} neurons, {g.session.nunique()} sessions, depths {g.depth.min()}-{g.depth.max()} um, r = {r:.3f}, p = {p:.2e}")
print("Paper: DT5 0.38, DT9 0.25, DT2 0.27, each p < 0.001.")
allc = sig.assign(depth=S.ls_depth[sig.session].values)
print("Variant, every significant condition:")
for subj in ["DT5", "DT9", "DT2"]:
    g = allc[allc.subject == subj]
    r, p = lshpc.circ_corrcl(g.onset_phase.values, g.depth.values)
    print(f"  {subj}: N = {len(g)}, r = {r:.3f}, p = {p:.2e}")

**Result: not reproduced.** The circular-linear correlation between onset phase and LS depth is r = 0.082 (p = 0.57) in DT5, 0.072 (p = 0.78) in DT9 and 0.118 (p = 0.65) in DT2, against 0.38, 0.25 and 0.27 at p < 0.001. Using every significant condition gives the same. The test rests on the session depth (the first number in the session name) and on a theta channel chosen per session, which may differ in layer between sessions.

## LS-hippocampal theta coherence along the run (fig6c-ca3-coherence-increases)

> Theta phase coherence between LS and the CA3 pyramidal layer increases from the beginning to the
> end of the run, while coherence between LS and CA1 stays relatively flat.

In [ ]:
rows = []
for i, r in enumerate(results):
    f6 = r.get("fig6")
    if not f6:
        continue
    sm = r["summary"]
    group = "CA3" if sm["has_ca3_units"] else "CA1"
    seg = np.arange(lshpc.N_SEGMENTS)
    ok = np.isfinite(f6["coherence"])
    ok2 = np.isfinite(f6["cfc"])
    rows.append({"session": i, "subject": sm["subject"], "group": group,
                 "coh_mean": np.nanmean(f6["coherence"]), "coh_slope": np.polyfit(seg[ok], f6["coherence"][ok], 1)[0] if ok.sum() > 2 else np.nan,
                 "cfc_mean": np.nanmean(f6["cfc"]), "cfc_slope": np.polyfit(seg[ok2], f6["cfc"][ok2], 1)[0] if ok2.sum() > 2 else np.nan,
                 "coh": f6["coherence"], "cfc": f6["cfc"]})
F6 = pd.DataFrame(rows)
print("Sessions by group and subject:")
print(F6.groupby(["group", "subject"]).size().to_string())
for g in ("CA3", "CA1"):
    x = F6[F6.group == g].coh_slope.dropna()
    w = stats.wilcoxon(x)
    print(f"{g}: N = {len(x)} sessions, median coherence slope {x.median():.4f} per segment, signed-rank p = {w.pvalue:.3f}, positive in {(x > 0).sum()}")
mw = stats.mannwhitneyu(F6[F6.group == "CA3"].coh_slope.dropna(), F6[F6.group == "CA1"].coh_slope.dropna(), alternative="greater")
print(f"CA3 slopes above CA1 slopes: Mann-Whitney p = {mw.pvalue:.3f}")
print("Mean coherence by segment:")
print(pd.DataFrame({g: np.nanmean(np.vstack(F6[F6.group == g].coh.values), 0) for g in ("CA3", "CA1")}).round(3).T.to_string())
print("Per subject, median slope:", F6.groupby(["group", "subject"]).coh_slope.median().round(4).to_dict())

**Result: not reproduced.** Theta phase locking between the hippocampal and LS channels does not rise along the run in the 11 CA3 sessions (median slope 0.0026 per segment, signed-rank p = 0.83) and CA3 slopes are not above CA1 slopes (p = 0.24). The measure is a phase locking value, not the paper's multitaper coherence, and the hippocampal channel is chosen here, not the paper's pyramidal layer site.

## Theta-gamma coupling between hippocampus and LS (fig6d-cfc-stronger-ca3)

> Coupling between hippocampal theta phase and LS gamma amplitude increases along the run for both
> CA1 and CA3, and is stronger for CA3-LS than for CA1-LS.

In [ ]:
mw = stats.mannwhitneyu(F6[F6.group == "CA3"].cfc_mean.dropna(), F6[F6.group == "CA1"].cfc_mean.dropna(), alternative="greater")
print(f"Mean coupling (circular-linear r): CA3 median {F6[F6.group == 'CA3'].cfc_mean.median():.4f}, CA1 median {F6[F6.group == 'CA1'].cfc_mean.median():.4f}; "
      f"CA3 above CA1, Mann-Whitney p = {mw.pvalue:.3f}")
for g in ("CA3", "CA1"):
    x = F6[F6.group == g].cfc_slope.dropna()
    w = stats.wilcoxon(x)
    print(f"{g}: N = {len(x)}, median slope {x.median():.5f} per segment, signed-rank p = {w.pvalue:.3f}, positive in {(x > 0).sum()}")
print("Mean coupling by segment:")
print(pd.DataFrame({g: np.nanmean(np.vstack(F6[F6.group == g].cfc.values), 0) for g in ("CA3", "CA1")}).round(4).T.to_string())
print("Per subject, mean coupling:", F6.groupby(["group", "subject"]).cfc_mean.median().round(4).to_dict())

**Result: reproduced.** Coupling between hippocampal theta phase and LS gamma amplitude is higher in CA3 sessions than in CA1 sessions (medians 0.059 and 0.049, Mann-Whitney p = 0.041) and rises along the run in both (signed-rank p = 0.002 and 0.001; positive in 10 of 11 and 35 of 55 sessions). The CA3 group is two rats, DT5 and DT9.

## Ensemble decoding (figs7-ensemble-ls-phase-best)

> The firing phases of ensembles of simultaneously recorded LS neurons read out position better than
> the firing phases or rates of an equal number of hippocampal neurons (p < 0.001 at 15 to 25 neurons).

In [ ]:
E = pd.DataFrame([dict(x, session=i, subject=results[i]["summary"]["subject"]) for i, r in enumerate(results) for x in r.get("ensemble", [])])
print("Conditions with at least 15 included LS and 15 hippocampal units:", E.drop_duplicates(["session", "condition"]).shape[0],
      "in", E.session.nunique(), "sessions;", E.drop_duplicates(["session", "condition"]).groupby("subject").size().to_dict())
for n, g in E.groupby("size"):
    v = {k: np.concatenate(g[k].values) for k in ("ls_phase", "ls_rate", "hpc_phase", "hpc_rate")}
    t1 = stats.ttest_ind(v["ls_phase"], v["hpc_phase"])
    t2 = stats.ttest_ind(v["ls_phase"], v["hpc_rate"])
    print(f"size {n}: {len(g)} conditions; mean MSE LS phase {v['ls_phase'].mean():.0f}, LS rate {v['ls_rate'].mean():.0f}, "
          f"HPC phase {v['hpc_phase'].mean():.0f}, HPC rate {v['hpc_rate'].mean():.0f}; LS phase vs HPC phase p = {t1.pvalue:.1e}, "
          f"vs HPC rate p = {t2.pvalue:.1e}")

**Result: not reproduced.** In 65 conditions from 13 sessions with at least 15 active LS and hippocampal units, ensembles of 15 hippocampal units decode position better from their rates (MSE 3,119) than LS ensembles do from their phases (4,738), and hippocampal phases (4,597) also do better than LS phases (p = 1.2e-03). Only at 25 units (3 conditions) are LS phases better than hippocampal phases (p = 0.015).

## Blocked claims: what the data allow

Hippocampal-LS cell assemblies (fig5-assemblies, and the claims that use them) are blocked as not
runnable. The count of simultaneously recorded pairs is printed here as a check of the dandiset
against the paper's 11,264 pairs.

In [ ]:
pairs = sum(len(set(LS[LS.session == s_].unit)) * len(set(HP[HP.session == s_].unit)) for s_ in S.session)
pairs_all = sum(d.get("ls", 0) * (d.get("hpc", 0) + d.get("ca3", 0)) for d in S.unit_locations)
print(f"Hippocampal-LS pairs of active units recorded together: {pairs}; of all labelled units: {pairs_all} (paper 11,264)")
print("Each pair needs 150 smoothing windows, each fitted to the data and to 10 trial shuffles (1,650 GLM fits per pair):",
      f"{pairs * 1650:,} fits for the active pairs.")

## Summary

| Claim | Outcome |
|---|---|
| ls-neuron-count | Reproduced |
| hpc-unit-count | Partially reproduced |
| ls-active-neurons | Reproduced |
| fig2-ls-phase-vs-shuffle | Not reproduced |
| fig2-ls-phase-beats-rate | Not reproduced |
| fig2e-ls-phase-lower-mse | Reproduced |
| fig2fg-mse-difference | Partially reproduced |
| fig1i-hpc-rate-phase-equivalent | Reproduced |
| ls-few-place-fields | Not reproduced |
| hpc-place-field-properties | Partially reproduced |
| ls-place-fields-unstable | Partially reproduced |
| fig1e-phase-info-smaller-scale | Partially reproduced |
| fig1g-phase-more-info-fine-scale | Partially reproduced |
| fig4d-depth-gradient | Partially reproduced |
| fig4e-onset-depth | Not reproduced |
| fig6c-ca3-coherence-increases | Not reproduced |
| fig6d-cfc-stronger-ca3 | Reproduced |
| figs7-ensemble-ls-phase-best | Not reproduced |
| fig5-assemblies, fig5g-assembly-neurons-stronger-phase, fig6e-assembly-position | Blocked: not runnable |
| speed-independent, ls-information-lower | Blocked: methods underspecified |
| figs6-reference-frame | Blocked: metadata insufficient |
| fig7-dynamic-weighting-model | Blocked: other |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.**

- The paper's linearized behaviour, trial conditions and choice of CA1 channel are not in the
  dandiset. Positions are linearized here by the fraction of each trial's path (200 bins, about 1.1
  cm), conditions are formed from trial type, direction and start and end sites, and the theta
  channel is the hippocampal channel with high theta and the most 100 to 250 Hz power.
- The per-cell decoder follows the paper's scratch code; the function that bins spike phases is not
  public, and the circular mean per bin is used.
- Place fields follow bz_getPlaceFields1D with the Methods' limits converted to bins at 1.5 cm.
  The call's own parameters are not public. This finds far fewer hippocampal fields than the paper.
- The information measure of Figure 1 is built from the Methods' formula with choices fixed in the
  claims file; the paper's function is not public.
- Coherence in Figure 6 is a phase locking value of the 4 to 12 Hz phases, not multitaper coherence.
- Only the trials are read from the LFP, with 1 s of padding for filtering.
- Assemblies (Figure 5, 6E) were not computed: about 21,000 hippocampal-LS pairs of active units
  would each need 1,650 GLM fits.

**What was found about the dandiset.**

- The paper's processed behaviour (linearized position and trial conditions), the CA1 channel per
  session, and the unit recording sites are not in the dandiset.
- Ten DT8 files state positions in mm but hold metres; in every file the z column of position is a
  copy of y; nine of eleven DT2 files repeat most position timestamps; DT2 positions are in pixels
  with no scale.
- DT2 electrodes are all labelled `unknown`; DT7 20170317 has 43 units with no region label; DT7
  20170324 has no units; 11 units hold spike times out of order.
- The depths in the session names are not described; the first is taken as the LS depth.
- DT9 20170501's trials do not match its positions; DT8 20170207 and 20170224 have trials without
  type or direction.
- The units labelled `hpc` (2,593) are 12.9% more than the paper's 2,297 CA1 units.